# Mental Health Dataset Audit

This notebook performs the first audit of the cleaned mental-health text dataset before preparing the data for the NLP experiments.

The audit checks:
- dataset size and structure
- missing values
- class distribution
- duplicate rows and duplicate texts
- duplicate texts with different labels
- text length statistics
- basic dataset characteristics


## 1. Load the dataset

In [1]:
import pandas as pd


DATA_PATH = "../data/cleaned_mental_health.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (55693, 2)


## 2. Basic dataset information

In [2]:
print("=" * 60)
print("MENTAL HEALTH DATASET AUDIT")
print("=" * 60)

print("\nDataset shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

MENTAL HEALTH DATASET AUDIT

Dataset shape:
(55693, 2)

Columns:
['label', 'text']

First five rows:


,label,text
0,Anxiety,calm Anxiety future
1,Anxiety,think religion caused Anxiety church specifica...
2,Anxiety,worst physical Anxiety symptom yall
3,Anxiety,paranoia ruining life made reddit account talk...
4,Anxiety,pre Anxiety guy ever feel disconnected pre Anx...


## 3. Missing values

Before training any model, we check whether the text or label columns contain missing values.

In [3]:
print("-" * 60)
print("MISSING VALUES")
print("-" * 60)

missing_values = df.isnull().sum()
display(missing_values.to_frame("missing_values"))

------------------------------------------------------------
MISSING VALUES
------------------------------------------------------------


,missing_values
label,0
text,0


## 4. Class distribution

The dataset contains multiple classification categories. We calculate both the number of examples and the percentage represented by each class.

In [4]:
print("-" * 60)
print("CLASS DISTRIBUTION")
print("-" * 60)

class_counts = df["label"].value_counts()
class_percentages = df["label"].value_counts(normalize=True) * 100

class_distribution = pd.DataFrame({
    "count": class_counts,
    "percentage": class_percentages.round(2)
})

display(class_distribution)

------------------------------------------------------------
CLASS DISTRIBUTION
------------------------------------------------------------


,count,percentage
label,,
Normal,16337,29.33
Depression,16167,29.03
Suicidal,11491,20.63
Anxiety,4808,8.63
Stress,2582,4.64
Mentalhealth,1962,3.52
Sad,1662,2.98
Happy,684,1.23


## 5. Duplicate analysis

Duplicate texts are important to identify before creating the train, validation, and test sets. If the same text appears in more than one split, it can lead to data leakage.

In [5]:
print("-" * 60)
print("DUPLICATES")
print("-" * 60)

duplicate_rows = df.duplicated().sum()
duplicate_texts = df["text"].duplicated().sum()

print("Duplicate rows:", duplicate_rows)
print("Duplicate texts:", duplicate_texts)

duplicate_texts_df = df[df["text"].duplicated(keep=False)].copy()

if not duplicate_texts_df.empty:
    label_counts = duplicate_texts_df.groupby("text")["label"].nunique()
    conflicting_duplicates = (label_counts > 1).sum()
else:
    conflicting_duplicates = 0

print("Duplicate texts with different labels:", conflicting_duplicates)

------------------------------------------------------------
DUPLICATES
------------------------------------------------------------
Duplicate rows: 1148
Duplicate texts: 1196
Duplicate texts with different labels: 47


## 6. Text length analysis

The number of whitespace-separated tokens is used here as a simple measure of text length. This will also help us later when discussing sequence length and truncation for the neural models.

In [6]:
print("-" * 60)
print("TEXT LENGTH")
print("-" * 60)

text_lengths = df["text"].fillna("").astype(str).str.split().str.len()

display(text_lengths.describe().to_frame("token_count"))

print("Empty texts:", (text_lengths == 0).sum())

------------------------------------------------------------
TEXT LENGTH
------------------------------------------------------------


,token_count
count,55693.000000
mean,106.104663
std,152.342449
min,1.000000
25%,15.000000
50%,58.000000
75%,136.000000
max,5248.000000


Empty texts: 0


## 7. Additional dataset checks

In [7]:
print("-" * 60)
print("ADDITIONAL CHECKS")
print("-" * 60)

print("Number of unique texts:", df["text"].nunique())
print("Number of unique labels:", df["label"].nunique())
            
print("\nLabels:")
print(sorted(df["label"].dropna().unique()))

------------------------------------------------------------
ADDITIONAL CHECKS
------------------------------------------------------------
Number of unique texts: 54497
Number of unique labels: 8

Labels:
['Anxiety', 'Depression', 'Happy', 'Mentalhealth', 'Normal', 'Sad', 'Stress', 'Suicidal']


## 8. Save the audit report

The audit results are saved as a text file so the dataset inspection can be kept with the experiment results.

In [8]:
from pathlib import Path


Path("results").mkdir(parents=True, exist_ok=True)

report_path = "results/dataset_audit.txt"

with open(report_path, "w", encoding="utf-8") as report:
    report.write("MENTAL HEALTH DATASET AUDIT\n")
    report.write("=" * 60 + "\n\n")

    report.write(f"Dataset shape: {df.shape}\n\n")

    report.write("Columns:\n")
    report.write(str(df.columns.tolist()) + "\n\n")

    report.write("Missing values:\n")
    report.write(str(df.isnull().sum()) + "\n\n")

    report.write("Class distribution:\n")
    report.write(str(class_distribution) + "\n\n")

    report.write(f"Duplicate rows: {duplicate_rows}\n")
    report.write(f"Duplicate texts: {duplicate_texts}\n")
    report.write(f"Duplicate texts with different labels: {conflicting_duplicates}\n\n")

    report.write("Text length statistics:\n")
    report.write(str(text_lengths.describe()) + "\n\n")

    report.write(f"Empty texts: {(text_lengths == 0).sum()}\n")
    report.write(f"Unique texts: {df['text'].nunique()}\n")
    report.write(f"Unique labels: {df['label'].nunique()}\n")

print("Audit complete.")
print(f"Report saved to: {report_path}")

Audit complete.
Report saved to: results/dataset_audit.txt


## Next step

After reviewing this audit, the next notebook will prepare a leakage-aware experimental dataset and create one fixed stratified train/validation/test split for all models.